# 03 — Sampling: temperature, top_p, top_k

<a href="https://colab.research.google.com/github/jorgeroa/ia-utn-frsf/blob/main/clase02/notebooks/03_sampling_params.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Objetivo.** Tocar los parámetros de sampling y ver cómo cambia el output. La idea es que después puedas elegirlos a conciencia para tu caso de uso.

**Requisitos.** API key de Groq en `GROQ_API_KEY`.


In [ ]:
%pip install --quiet groq


In [ ]:
import os
from groq import Groq

try:
    from google.colab import userdata
    os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
except ImportError:
    assert os.environ.get("GROQ_API_KEY"), "Exportá GROQ_API_KEY."

client = Groq()
MODEL = "qwen/qwen3-32b"

def generar(prompt, **kwargs):
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        **kwargs,
        # reasoning_effort="none",  # descomentar para que Qwen3 no devuelva <think>...</think>
    )
    return resp.choices[0].message.content


## 1. `temperature` — el termostato de la creatividad

Mismo prompt, distintas temperaturas. Observá la diferencia de tono y vocabulario.


In [ ]:
PROMPT = "Escribime un poema corto (4 versos) sobre el otoño en Buenos Aires."

for temp in [0.0, 0.3, 0.7, 1.2]:
    print(f"--- temperature = {temp} ---")
    print(generar(PROMPT, temperature=temp))
    print()


- `temperature=0` → el modelo elige siempre el token más probable. Determinista.
- Valores altos → distribución más plana, salidas diversas (y a veces incoherentes).


## 2. Reproducibilidad: el bug del "siempre lo mismo"

Con temperatura baja, dos llamadas seguidas dan respuestas casi idénticas. Útil cuando necesitás determinismo (testing, código).


In [ ]:
PROMPT = "Listame 3 razones por las que se prefiere PostgreSQL sobre MySQL."

for i in range(2):
    print(f"--- Llamada {i+1} (temperature=0) ---")
    print(generar(PROMPT, temperature=0))
    print()


## 3. `top_p` — nucleus sampling

Muestrea solo del conjunto de tokens cuya probabilidad acumulada llega a P. Recorta la "cola larga".


In [ ]:
PROMPT = "Inventame el nombre de una banda de rock progresivo argentina."

for p in [0.1, 0.5, 1.0]:
    print(f"--- top_p = {p} ---")
    for _ in range(3):
        print("  ·", generar(PROMPT, temperature=0.9, top_p=p))
    print()


## 4. El bug del repetition loop

Con `temperature` muy baja **y** un prompt ambiguo, el modelo puede entrar en bucle repitiendo frases. Lo provocamos a propósito:


In [ ]:
# Prompt cuasi-vacío + temperatura mínima -> riesgo de loop
print(generar("repetí: ", temperature=0.0, max_tokens=100))


## Cuándo usar qué

| Caso | temperature | top_p |
|---|---|---|
| Código, factual, extracción | 0.0 – 0.3 | 1.0 |
| Conversación natural | 0.6 – 0.8 | 0.9 |
| Creatividad, brainstorming | 0.9 – 1.2 | 0.95 |
| Determinismo (tests) | 0.0 | 1.0 |
